# PCA with pyphi

Builds a PCA model via NIPALS, interprets scores/loadings, and uses control charts for outlier detection.

In [1]:
import pandas as pd
import numpy as np
import pyphi.calc as phi
import pyphi.plots as pp
from bokeh.io import output_notebook
output_notebook(hide_banner=True)
import pyphi.plots as _ppmod
_ppmod.output_file = lambda *a, **kw: None


Will be using the NEOS server in the absence of IPOPT and GAMS


## Load Data

In [2]:
features = pd.read_excel('../data/Automobiles PCA w MD.xlsx', 'Features',
                         na_values=np.nan, engine='openpyxl')
classid  = pd.read_excel('../data/Automobiles PCA w MD.xlsx', 'CLASSID',
                         na_values=np.nan, engine='openpyxl')
print(features.shape)
features.head()


(406, 6)


,Car Number,Cylinders,Displacement,Horsepower,Model Year,Weight
0,Car1,4.0,122.0,88,80.0,2500.0
1,Car2,4.0,133.0,115,70.0,3090.0
2,Car3,NaN,110.0,87,70.0,2672.0
3,Car4,4.0,79.0,70,71.0,2074.0
4,Car5,4.0,120.0,87,72.0,2979.0


## Build PCA Model

`mcs=True` (default) autoscales. Options: `'center'`, `'autoscale'`, `False`. `cross_val=5` holds out 5% of elements per CV round.

In [3]:
pcaobj = phi.pca(features, 3, cross_val=5)
print('Keys:', list(pcaobj.keys()))
print('T shape:', pcaobj['T'].shape, '  P shape:', pcaobj['P'].shape)
print('r2x (cumulative):', pcaobj['r2x'])
print('q2 (cross-val) :', pcaobj['q2'])


Cross validating PC #1


Cross validating PC #2


Cross validating PC #3


phi.pca using NIPALS and cross validation (5%) executed on: 2026-03-26 23:35:23.410293
--------------------------------------------------------------
PC #          Eig      R2X     sum(R2X)      Q2X     sum(Q2X)
PC #1:      3.930    0.771     0.771       0.659     0.659
PC #2:      1.129    0.173     0.944       0.158     0.816
PC #3:      0.244    0.031     0.975       -1504.743     -1503.926
--------------------------------------------------------------
Keys: ['T', 'P', 'r2x', 'r2xpv', 'mx', 'sx', 'var_t', 'obsidX', 'varidX', 'T2', 'T2_lim99', 'T2_lim95', 'speX', 'speX_lim99', 'speX_lim95', 'q2', 'q2pv', 'type']
T shape: (406, 3)   P shape: (5, 3)
r2x (cumulative): [0.77096198 0.17293508 0.03090922]
q2 (cross-val) : [ 6.58890911e-01  1.57592634e-01 -1.50474269e+03]


## Captured Variance per Variable per Component

In [4]:
pp.r2pv(pcaobj)

## Score Plots

In [5]:
pp.score_scatter(pcaobj, [1, 2], CLASSID=classid, colorby='Origin')
pp.score_scatter(pcaobj, [1, 2], CLASSID=classid, colorby='Cylinders')
pp.score_line(pcaobj, 1, CLASSID=classid, colorby='Origin', add_ci=True, add_labels=True)


## Loadings

In [6]:
pp.loadings(pcaobj)
pp.weighted_loadings(pcaobj)
pp.loadings_map(pcaobj, [1, 2])


## Diagnostics: Hotelling's T² and SPE

Points beyond `T2_lim99` or `speX_lim99` are statistical outliers.

In [7]:
pp.diagnostics(pcaobj, score_plot_xydim=[1, 2])
print('T2  limits (95/99):', pcaobj['T2_lim95'],  pcaobj['T2_lim99'])
print('SPE limits (95/99):', pcaobj['speX_lim95'], pcaobj['speX_lim99'])


T2  limits (95/99): 7.939762549663908 11.577125732240635
SPE limits (95/99): 0.554694001229005 1.1165412163822814


## Contribution Plots

`'scores'` mode shows which variables drive the difference between two observations.

In [8]:
pp.contributions_plot(pcaobj, features, 'scores', to_obs=['Car1'])
pp.contributions_plot(pcaobj, features, 'scores', to_obs=['Car1'], from_obs=['Car4'])
